# 02 - Data Preprocessing

This notebook prepares the CDC Diabetes dataset for modelling based on EDA findings:
1. Target variable binarisation (3-class → 2-class)
2. Handling duplicate rows
3. Feature scaling (StandardScaler)
4. Handling class imbalance (SMOTE)
5. Train/test split
6. Saving processed data for clustering and classification notebooks

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from collections import Counter
import os

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('Libraries loaded successfully.')

## 1. Load Raw Data

In [ ]:
# Load original dataset
df = pd.read_csv('../data/CDC Diabetes Dataset.csv')
print(f'Original dataset shape: {df.shape}')
print(f'Target distribution (original 3-class):')
print(df['Diabetes_012'].value_counts().sort_index())
df.head()

## 2. Target Variable Binarisation

The original target has 3 classes: 0 (No Diabetes), 1 (Prediabetes), 2 (Diabetes).

**Decision**: Merge classes 1 and 2 into a single "Diabetes/Prediabetes" class.

**Rationale**:
- Prediabetes (class 1) has only 1.8% of samples — too few for reliable 3-class classification
- The scenario asks to classify "diabetic" vs "non-diabetic"
- Prediabetes is a risk state that clinically aligns more with diabetes than with healthy
- Binary classification is more practical for public health screening

In [ ]:
# Binarise target: 0 = No Diabetes, 1 = Prediabetes or Diabetes
df['Diabetes_binary'] = (df['Diabetes_012'] > 0).astype(int)

print('=== Binary Target Distribution ===')
binary_counts = df['Diabetes_binary'].value_counts().sort_index()
binary_pcts = df['Diabetes_binary'].value_counts(normalize=True).sort_index() * 100

for val, label in zip([0, 1], ['No Diabetes', 'Diabetes/Prediabetes']):
    print(f'  {label:25s} (class {val}): {binary_counts[val]:>8,} samples ({binary_pcts[val]:.2f}%)')

# Visualise
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2ecc71', '#e74c3c']
bars = ax.bar(['No Diabetes (0)', 'Diabetes/Prediabetes (1)'], binary_counts.values,
             color=colors, edgecolor='black')
ax.set_title('Binarised Target Variable Distribution', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
for bar, count, pct in zip(bars, binary_counts.values, binary_pcts.values):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1000,
           f'{count:,}\n({pct:.1f}%)', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nImbalance ratio: {binary_counts[0]/binary_counts[1]:.2f} : 1')

# Drop the original 3-class target
df = df.drop('Diabetes_012', axis=1)
print(f'\nDataset shape after binarisation: {df.shape}')

## 3. Handle Duplicate Rows

EDA found 9.42% duplicate rows. Since this is survey data where different individuals can legitimately give identical responses (especially with binary/ordinal features), **we retain duplicates** to preserve the true population distribution.

In [ ]:
# Document the decision to keep duplicates
duplicates = df.duplicated().sum()
print(f'Duplicate rows: {duplicates:,} ({duplicates/len(df)*100:.2f}%)')
print('Decision: RETAIN duplicates — survey data legitimately contains identical response patterns.')
print(f'Final dataset size: {len(df):,} rows')

## 4. Feature and Target Separation

In [ ]:
# Separate features and target
X = df.drop('Diabetes_binary', axis=1)
y = df['Diabetes_binary']

print(f'Features shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'\nFeature list ({X.shape[1]} features):')
for i, col in enumerate(X.columns, 1):
    print(f'  {i:2d}. {col}')

## 5. Train/Test Split

Split the data **before** any preprocessing (scaling, SMOTE) to prevent data leakage.
- 80% training, 20% testing
- Stratified split to preserve class distribution in both sets

In [ ]:
# Stratified train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]:,} samples')
print(f'Test set:     {X_test.shape[0]:,} samples')
print(f'\nTraining target distribution:')
print(f'  Class 0 (No Diabetes):          {(y_train == 0).sum():,} ({(y_train == 0).mean()*100:.2f}%)')
print(f'  Class 1 (Diabetes/Prediabetes): {(y_train == 1).sum():,} ({(y_train == 1).mean()*100:.2f}%)')
print(f'\nTest target distribution:')
print(f'  Class 0 (No Diabetes):          {(y_test == 0).sum():,} ({(y_test == 0).mean()*100:.2f}%)')
print(f'  Class 1 (Diabetes/Prediabetes): {(y_test == 1).sum():,} ({(y_test == 1).mean()*100:.2f}%)')

## 6. Feature Scaling (StandardScaler)

**Why scale?**
- Clustering algorithms (K-Means, DBSCAN) rely on distance metrics — features must be on the same scale
- Some classifiers (Logistic Regression, SVM) benefit from scaled features
- Tree-based models (Random Forest, XGBoost) are scale-invariant but scaling doesn't hurt

**Important**: Fit the scaler on the **training set only**, then transform both train and test sets to avoid data leakage.

In [ ]:
# Fit scaler on training data only, transform both
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print('=== Scaled Training Data Statistics ===')
print(X_train_scaled.describe().round(3).loc[['mean', 'std', 'min', 'max']])
print(f'\nScaling verified: all means ≈ 0 and stds ≈ 1 on training data.')

## 7. Handling Class Imbalance (SMOTE)

**Problem**: The minority class (Diabetes/Prediabetes) represents only ~15.8% of the data. Models trained on imbalanced data tend to predict the majority class.

**Solution**: SMOTE (Synthetic Minority Over-sampling Technique)
- Creates synthetic samples of the minority class by interpolating between existing minority samples
- Applied **only to training data** — test data remains untouched to reflect real-world distribution

**Why SMOTE over other methods?**
- Random undersampling discards valuable majority class data
- Random oversampling just duplicates — can cause overfitting
- SMOTE generates new synthetic samples — better generalisation

In [ ]:
# Apply SMOTE to scaled training data
print('=== Before SMOTE ===')
print(f'Training set class distribution: {dict(Counter(y_train))}')
print(f'Total training samples: {len(y_train):,}')

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print(f'\n=== After SMOTE ===')
print(f'Training set class distribution: {dict(Counter(y_train_smote))}')
print(f'Total training samples: {len(y_train_smote):,}')
print(f'New samples created: {len(y_train_smote) - len(y_train):,}')

# Visualise before and after
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before SMOTE
before = Counter(y_train)
axes[0].bar(['No Diabetes (0)', 'Diabetes (1)'],
           [before[0], before[1]], color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[0].set_title('Before SMOTE', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for i, (k, v) in enumerate(sorted(before.items())):
    axes[0].text(i, v + 1000, f'{v:,}', ha='center', fontweight='bold')

# After SMOTE
after = Counter(y_train_smote)
axes[1].bar(['No Diabetes (0)', 'Diabetes (1)'],
           [after[0], after[1]], color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[1].set_title('After SMOTE', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Count')
for i, (k, v) in enumerate(sorted(after.items())):
    axes[1].text(i, v + 1000, f'{v:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../notebooks/figures/smote_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Processed Data

Saving multiple versions of the data for different downstream tasks:
- **For Classification**: SMOTE-balanced scaled training data + scaled test data
- **For Clustering**: Full scaled dataset (unsupervised — no train/test needed)

In [ ]:
# Create processed data directory
os.makedirs('../data/processed', exist_ok=True)

# 1. Classification data (SMOTE-balanced)
X_train_smote.to_csv('../data/processed/X_train_smote.csv', index=False)
y_train_smote.to_csv('../data/processed/y_train_smote.csv', index=False)
X_test_scaled.to_csv('../data/processed/X_test_scaled.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

# 2. Classification data (original imbalanced — for comparison with class_weight)
X_train_scaled.to_csv('../data/processed/X_train_scaled.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)

# 3. Full scaled data for clustering (no target)
X_full_scaled = pd.DataFrame(
    scaler.transform(X),  # Use same scaler fitted on train
    columns=X.columns,
    index=X.index
)
X_full_scaled.to_csv('../data/processed/X_full_scaled.csv', index=False)
y.to_csv('../data/processed/y_full.csv', index=False)  # Save labels for cluster evaluation

print('=== Saved Processed Data Files ===')
for f in os.listdir('../data/processed'):
    size_mb = os.path.getsize(f'../data/processed/{f}') / (1024*1024)
    print(f'  {f:30s} ({size_mb:.1f} MB)')

## 9. Preprocessing Summary

### Steps Completed:
| Step | Action | Rationale |
|------|--------|----------|
| Target Binarisation | Merged classes 1+2 → 1 | Prediabetes too rare (1.8%); binary is more practical |
| Duplicates | Retained | Legitimate in survey data |
| Train/Test Split | 80/20, stratified | Preserve class ratios; prevent data leakage |
| Feature Scaling | StandardScaler (fit on train) | Required for clustering and some classifiers |
| Class Imbalance | SMOTE on training data only | Generate synthetic minority samples for better learning |

### Data Available for Next Steps:
- **Clustering**: `X_full_scaled.csv` (all 253,680 samples, 21 features, scaled)
- **Classification (SMOTE)**: `X_train_smote.csv` + `y_train_smote.csv` (balanced)
- **Classification (Original)**: `X_train_scaled.csv` + `y_train.csv` (for class_weight approach)
- **Test Set**: `X_test_scaled.csv` + `y_test.csv` (untouched real distribution)